In [10]:
pip install pandas lxml osmnx

Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd
import requests
from io import StringIO

URL = "https://www.emol.com/noticias/Nacional/2025/04/29/1164920/indice-calidad-vida-comunas-variables.html"
html = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text
tablas = pd.read_html(StringIO(html))

for i, t in enumerate(tablas):
    print(i, t.shape)      # deberían ver una tabla de (99, 8)

0 (99, 8)


In [14]:
import osmnx as ox

lugar = "Providencia, Santiago, Chile"

# 1. El límite
limite = ox.geocode_to_gdf(lugar)
print(limite)

# 2. Elementos por tag
parques = ox.features_from_place(lugar, tags={"leisure": "park"})
print(parques.columns)

# 3. La red vial
G = ox.graph_from_place(lugar, network_type="drive")
nodos, aristas = ox.graph_to_gdfs(G)

                                            geometry  bbox_west  bbox_south  \
0  POLYGON ((-70.63626 -33.43092, -70.63601 -33.4... -70.636261  -33.449837   

   bbox_east  bbox_north  place_id  osm_type  osm_id        lat        lon  \
0 -70.582252  -33.408136   1697606  relation  162983 -33.428838 -70.611337   

      class            type  place_rank  importance addresstype         name  \
0  boundary  administrative          16     0.49782        city  Providencia   

                                        display_name  
0  Providencia, Provincia de Santiago, Santiago M...  
Index(['geometry', 'addr:street', 'leisure', 'name', 'operator', 'addr:city',
       'operator:type', 'addr:country', 'object:street', 'old_name', 'access',
       'opening_hours', 'addr:postcode', 'alt_name', 'check_date', 'dog',
       'lit', 'loc_name', 'nohousenumber', 'start_date', 'surface', 'website',
       'wikidata', 'wikimedia_commons', 'wikipedia', 'source', 'architect',
       'contact:facebook', 

In [15]:
import osmnx as ox
import pandas as pd

lugar = "Providencia, Santiago, Chile"

# Claves que quieres explorar (True = cualquier valor)
tags = {
    "building": True,
    "amenity": True,
    "shop": True,
    "leisure": True,
    "tourism": True,
    "office": True,
}

features = ox.features_from_place(lugar, tags=tags)
features = features.reset_index()   # element e id pasan a ser columnas

# Lista de diccionarios: una estructura = un diccionario con sus tags no vacíos
estructuras = [
    {k: v for k, v in fila.items() if k != "geometry" and pd.notna(v)}
    for _, fila in features.iterrows()
]

print(len(estructuras))
print(estructuras[0])

16518
{'element': 'node', 'id': 133663460, 'alt_name': 'Ciudadano Restaurant', 'amenity': 'pub', 'name': 'Bar Ciudadano', 'website': 'https://www.ciudadanorestaurant.cl/'}


In [18]:
import osmnx as ox
import pandas as pd
import geopandas as gpd
from osmnx._errors import InsufficientResponseError

ox.settings.log_console = False
ox.settings.use_cache = True
ox.settings.requests_timeout = 300
ox.settings.overpass_rate_limit = True   # respeta el límite del servidor

LUGAR = "Providencia, Santiago, Chile"

categorias = {
    "Salud": {
        "Hospital":        {"amenity": "hospital"},
        "Clinica":         {"amenity": "clinic"},
        "Centro_salud":    {"healthcare": "centre"},          # CESFAM, etc.
        "Consultorio":     {"amenity": "doctors"},
        "Farmacia":        {"amenity": "pharmacy"},
    },
    "Recreativo_Cultural": {
        "Parque":          {"leisure": "park"},
        "Mall":            {"shop": "mall"},
        "Restaurante":     {"amenity": "restaurant"},
        "Museo":           {"tourism": "museum"},
        "Biblioteca":      {"amenity": "library"},
        "Teatro":          {"amenity": "theatre"},
    },
    "Seguridad": {
        "Bomberos":        {"amenity": "fire_station"},
        "Comisaria":       {"amenity": "police"},
    },
    "Educacion": {
        "Colegio":         {"amenity": "school"},
        "Jardin_infantil": {"amenity": "kindergarten"},
        "Universidad":     {"amenity": "university"},
    },
    "Comercio": {
        "Supermercado":    {"shop": "supermarket"},
        "Banco":           {"amenity": "bank"},
        "Feria":           {"amenity": "marketplace"},
        "Almacen":         {"shop": ["convenience", "general", "greengrocer"]},
    },
    "Transporte": {
        "Metro_estacion":  {"station": "subway"},
        "Metro_railway":   {"railway": "station"},            # incluye EFE; filtrar luego
        "Metro_entrada":   {"railway": "subway_entrance"},
        "Paradero_bus":    {"highway": "bus_stop"},
        "Paradero_plataforma": {"public_transport": "platform"},
        "Terminal_bus":    {"amenity": "bus_station"},
    },
    "Negativos": {
        "Vertedero":       {"landuse": "landfill"},
        "Planta_residuos": {"amenity": "waste_transfer_station"},
        "Industrial":      {"landuse": "industrial"},
        "Fabrica":         {"man_made": "works"},
        "Botilleria":      {"shop": "alcohol"},
        "Bar_pub":         {"amenity": ["bar", "pub", "nightclub"]},
        "Sitio_eriazo":    {"landuse": ["brownfield", "greenfield"]},
        "Suelo_desnudo":   {"natural": "bare_rock"},
    },
}

resultados, resumen = [], []

for categoria, subtipos in categorias.items():
    for tipo, tags in subtipos.items():
        try:
            gdf = ox.features_from_place(LUGAR, tags=tags)
        except Exception as e:      # sin resultados o error de red
            print(f"[0] {categoria}/{tipo}: {type(e).__name__}")
            resumen.append((categoria, tipo, 0))
            continue

        gdf = gdf.reset_index()                  # element, id -> columnas
        gdf = gdf.rename(columns={"element": "osm_type", "id": "osm_id"})
        gdf["categoria"] = categoria
        gdf["tipo"] = tipo
        gdf["query_tags"] = str(tags)

        resultados.append(gdf)
        resumen.append((categoria, tipo, len(gdf)))
        print(f"[{len(gdf)}] {categoria}/{tipo}")

# Unión de todo (columnas distintas -> NaN donde no aplica)
datos = gpd.GeoDataFrame(
    pd.concat(resultados, ignore_index=True), geometry="geometry", crs="EPSG:4326"
)

# Coordenadas representativas (centroide calculado en CRS proyectado)
cent = datos.to_crs(32719).centroid.to_crs(4326)
datos["lon"], datos["lat"] = cent.x, cent.y
datos["geom_tipo"] = datos.geom_type

# Límite de la comuna
limite = ox.geocode_to_gdf(LUGAR)

# Resumen de conteos
df_resumen = pd.DataFrame(resumen, columns=["categoria", "tipo", "n"])
print(df_resumen)

# Guardado
datos.to_file("providencia_features.gpkg", layer="features", driver="GPKG")
limite.to_file("providencia_features.gpkg", layer="limite", driver="GPKG")
datos.drop(columns="geometry").to_csv("providencia_features.csv", index=False)
df_resumen.to_csv("resumen_conteos.csv", index=False)

[0] Salud/Hospital: ConnectionError


KeyboardInterrupt: 